In [2]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [8]:
files = glob('gemini-flash-2.0-speech/*/*.parquet')
len(files)

99

In [3]:
df = pd.read_parquet(files[0])
df

,kore,puck,text,phoneme_length
0,{'bytes': b'RIFF$:\x02\x00WAVEfmt \x10\x00\x00...,{'bytes': b'RIFF$\x94\x02\x00WAVEfmt \x10\x00\...,"Unfortunately, I do not have an answer to that...",58.0
1,{'bytes': b'RIFF$h\x01\x00WAVEfmt \x10\x00\x00...,{'bytes': b'RIFF\xa4`\x01\x00WAVEfmt \x10\x00\...,We sip in silence for a while.,31.0
2,{'bytes': b'RIFF\xa4`\x10\x00WAVEfmt \x10\x00\...,{'bytes': b'RIFF$\xe2\x0e\x00WAVEfmt \x10\x00\...,"It says of itself: ""Earthsharing Australia rep...",344.0
3,{'bytes': b'RIFF\xa4/\x05\x00WAVEfmt \x10\x00\...,{'bytes': b'RIFF\xa4\xa8\x04\x00WAVEfmt \x10\x...,"So. Yeah. Uh, so I like that, uh, a good richn...",79.0
4,{'bytes': b'RIFF$L\x0e\x00WAVEfmt \x10\x00\x00...,{'bytes': b'RIFF\xa4T\r\x00WAVEfmt \x10\x00\x0...,The fine wisps clipped from his infant daughte...,318.0
...,...,...,...,...
472,{'bytes': b'RIFF\xa4}\x11\x00WAVEfmt \x10\x00\...,{'bytes': b'RIFF$\xef\x10\x00WAVEfmt \x10\x00\...,The forest trees were painted in richest hue o...,386.0
473,{'bytes': b'RIFF\xa4\xd6\x03\x00WAVEfmt \x10\x...,{'bytes': b'RIFF\xa4{\x04\x00WAVEfmt \x10\x00\...,It models burst noise (also called popcorn noi...,83.0
474,{'bytes': b'RIFF$\xfc\x03\x00WAVEfmt \x10\x00\...,{'bytes': b'RIFF\xa4\x99\x04\x00WAVEfmt \x10\x...,Content moderation questions. It doesn't make ...,102.0
475,{'bytes': b'RIFF$\x8f\x07\x00WAVEfmt \x10\x00\...,{'bytes': b'RIFF\xa4Z\x07\x00WAVEfmt \x10\x00\...,"Now, that was just my personal experience, but...",197.0


In [9]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = '_'.join(f.split('/')[:2]) + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
                
            audio_filename = f'{f_new}_{i}_kore.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['kore'].iloc[i]['bytes']
            sr, audio_np = wavfile.read(io.BytesIO(b))
            sf.write(audio_filename, audio_np, sr)

            data.append({
                'audio_filename': audio_filename,
                'text': df['text'].iloc[i],
                'speaker': f"{base}_kore"
            })

            audio_filename = f'{f_new}_{i}_puck.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['puck'].iloc[i]['bytes']
            sr, audio_np = wavfile.read(io.BytesIO(b))
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': df['text'].iloc[i],
                'speaker': f"{base}_puck"
            })
        
    return data

In [10]:
data = loop((files[:1], 0))

100%|██████████| 477/477 [02:36<00:00,  3.06it/s]


In [ ]:
data = multiprocessing(files, loop, cores = min(10, len(files)))

 37%|███▋      | 178/477 [00:59<01:53,  2.64it/s]

In [12]:
len(data)

94504

In [13]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'gemini-flash-2.0-speech_data_audio/gemini-flash-2.0-speech-data-en-00039-of-00099_0_kore.mp3',
 'text': 'Unfortunately, I do not have an answer to that question.',
 'speaker': 'gemini-flash-2.0-speech_data_audio_kore'}

In [14]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'gemini-flash-2.0-speech')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 20.62ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1): 100%|█████████▉| 7.45MB / 7.48MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 7.48MB / 7.48MB,  269kB/s  
Processing Files (1 / 1): 100%|██████████| 7.48MB / 7.48MB,  147kB/s  
New Data Upload: 100%|██████████| 7.48MB / 7.48MB,  147kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.38 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/e374ca817ec94874585bd298c5f71112959dba22', commit_message='Upload dataset', commit_description='', oid='e374ca817ec94874585bd298c5f71112959dba22', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [15]:
audio_files = [d['audio_filename'] for d in data]

with open('gemini-flash-2.0-speech-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [3]:
folders = glob('gemini-flash-2.0-speech_data_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

gemini-flash-2.0-speech_data_audio_neucodec
gemini-flash-2.0-speech_data_audio


In [4]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('gemini-flash-2.0-speech_data_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|          | 23.1MB / 8.07GB,   ???B/s  
Processing Files (0 / 1):   2%|▏         |  123MB / 8.07GB,  499MB/s  
Processing Files (0 / 1):   3%|▎         |  257MB / 8.07GB,  584MB/s  
Processing Files (0 / 1):   5%|▍         |  367MB / 8.07GB,  573MB/s  
Processing Files (0 / 1):   6%|▌         |  500MB / 8.07GB,  595MB/s  
Processing Files (0 / 1):   7%|▋         |  600MB / 8.07GB,  577MB/s  
Processing Files (0 / 1):   9%|▉         |  710MB / 8.07GB,  573MB/s  
Processing Files (0 / 1):  10%|█         |  840MB / 8.07GB,  584MB/s  
Processing Files (0 / 1):  12%|█▏        |  969MB / 8.07GB,  591MB/s  
Processing Files (0 / 1):  13%|█▎       